[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/099.%20The%20Electric%20Vehicle%20Routing%20Problem%20%28EVRP%29/P99-Tier-8_executed.ipynb)

---


# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 8: Value-Aligned and Ethical Framework

### Purpose and scope

This example builds a small, transparent EV route-construction heuristic that accounts for social importance and community sensitivity. It then evaluates the constructed routes using stakeholder utility functions and a set of ethical-compliance scores. The example is intended to teach how value measures can be represented and reported alongside routing decisions; it does **not** solve the general EV routing problem to global optimality.

**Inputs:** one depot, six customer locations, two charging-station locations, two vehicles, battery and energy parameters, stakeholder weights and preferences, and illustrative ethical thresholds.

**Outputs:** up to two depot-to-depot routes, route and stakeholder metrics, ethical-compliance scores, a comparison with a constructed illustrative reference scenario, visualizations, and an audit report.

### Modeling assumptions

- Locations are points in a two-dimensional plane. Euclidean distance is used, and one coordinate unit represents one kilometre.
- Each route starts and ends at the depot. A vehicle must retain a 20 kWh reserve after returning to the depot. Charging stations are shown on the map but are **not** visited or used by the route-construction procedure.
- Proximity to a charging station is reported only as a renewable-charging opportunity proxy. It is not a measurement of energy actually charged or consumed from renewable sources.
- Energy consumption is constant at 0.8 kWh per kilometre. Emissions and monetary cost use simplified illustrative rates; they are not a life-cycle assessment or a market tariff.
- Stakeholder preferences and several operational indicators are illustrative inputs. Compliance scores are normalized indicators, not legal certifications or objective measurements of ethical quality.
- The model does not include time windows, customer demand quantities, traffic, partial charging, charging duration, vehicle differences, or uncertainty.

### Method and mathematical representation

The route constructor repeatedly selects the feasible unserved customer with the highest ethical score. For a current location \(i\) and candidate customer \(j\), the score is

$$S(i,j) = 0.4 I_j + 0.3\frac{1}{1+d(i,j)} + 0.3(1-C_j),$$

where \(I_j\) is the customer's social-importance value, \(C_j\) is community sensitivity, and \(d(i,j)\) is Euclidean distance in kilometres. The three terms reward social importance, shorter travel, and lower community sensitivity. A candidate is feasible only if travel to it and a direct return to the depot preserve the 20 kWh reserve. The heuristic serves customers in score order; it does not search all possible route permutations and makes no global-optimality guarantee.

For stakeholder \(s\), let \(u_s(x)\) be the utility function applied to the indicators relevant to that stakeholder, and let \(w_s\) be its weight. The weighted utility before ethical penalties is

$$U = \sum_{s\in\mathcal{S}} w_s u_s(x_s).$$

If compliance for principle \(p\) is \(q_p\), its threshold is \(\tau_p\), and its penalty weight is \(\lambda_p\), the configured soft-constraint penalty is

$$P = \sum_{p\in\mathcal{P}} \lambda_p\max(0,\tau_p-q_p), \qquad U_{\mathrm{adjusted}}=U-P.$$

Here, \(\mathcal{S}\) is the stakeholder set and \(\mathcal{P}\) is the set of configured ethical constraints. The code implements this weighted-utility calculation and penalty in `StakeholderUtilityOptimizer` and `ConstitutionalAIFramework`. The route-construction heuristic and stakeholder scoring are separate stages: the scores evaluate and describe the constructed solution; they are not an exact multi-objective routing optimizer.

### Workflow

1. Define ethical principles, soft thresholds, and stakeholder utility functions.
2. Construct routes using the score-based heuristic while checking battery feasibility.
3. Calculate route metrics and stakeholder utilities, then apply the configured ethical penalty.
4. Compare the result with a clearly labeled, synthetic utilitarian reference scenario.
5. Visualize the routes and indicators, and create an audit report.

### Interpreting the results

A higher utility or compliance score means a higher value under the stated illustrative scoring rules; it does not establish that a route is ethically acceptable in every real-world context. Scores depend on the chosen normalization, stakeholder weights, proxy indicators, and assumptions. The reference comparison is a constructed scenario, not the output of a second routing optimization.

In [ ]:
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

np.random.seed(42)


class EthicalPrinciple(Enum):
    ENVIRONMENTAL_SUSTAINABILITY = "environmental_sustainability"
    SOCIAL_EQUITY = "social_equity"
    ECONOMIC_VIABILITY = "economic_viability"
    TRANSPARENCY = "transparency"
    ACCOUNTABILITY = "accountability"
    FAIRNESS = "fairness"
    SAFETY = "safety"
    PRIVACY = "privacy"


class StakeholderType(Enum):
    CUSTOMER = "customer"
    ENVIRONMENT = "environment"
    COMMUNITY = "community"
    EMPLOYEE = "employee"
    INVESTOR = "investor"
    REGULATOR = "regulator"
    SOCIETY = "society"


@dataclass
class Stakeholder:
    """A stakeholder, its utility function, preferences, and decision weight."""

    stakeholder_id: int
    stakeholder_type: StakeholderType
    name: str
    utility_function: str
    weight: float
    preferences: Dict[str, float] = field(default_factory=dict)
    constraints: List[str] = field(default_factory=list)
    ethical_principles: List[EthicalPrinciple] = field(default_factory=list)


@dataclass
class EthicalConstraint:
    """A threshold and penalty rule associated with an ethical principle."""

    constraint_id: int
    principle: EthicalPrinciple
    description: str
    constraint_type: str
    threshold: float
    penalty_weight: float = 1.0
    measurement_function: str = "linear"


@dataclass
class ValueAlignedSolution:
    """A constructed route plan and its value and compliance assessments."""

    routes: List[List[Tuple[float, float]]]
    stakeholder_utilities: Dict[int, float]
    ethical_compliance: Dict[EthicalPrinciple, float]
    total_utility: float
    environmental_impact: float
    social_impact: float
    economic_cost: float
    transparency_score: float
    accountability_metrics: Dict[str, float]
    ethical_reasoning: List[str]


@dataclass
class EthicalAudit:
    """A reproducible-in-content audit record for a solution assessment."""

    audit_id: str
    timestamp: datetime
    solution: ValueAlignedSolution
    compliance_score: float
    violations: List[str]
    stakeholder_feedback: Dict[int, str]
    recommendations: List[str]
    ethical_score_improvement: float

In [ ]:
class ConstitutionalAIFramework:
    """Stores ethical principles and evaluates illustrative compliance indicators."""

    def __init__(self):
        self.constitution = {
            EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: {
                "description": "Limit environmental burdens and support sustainability.",
                "constraints": [
                    "Prefer renewable charging opportunities.",
                    "Limit energy consumption and emissions.",
                    "Avoid environmentally sensitive areas.",
                ],
                "metrics": ["co2_emissions", "energy_efficiency", "renewable_opportunity"],
            },
            EthicalPrinciple.SOCIAL_EQUITY: {
                "description": "Support fair service across communities.",
                "constraints": [
                    "Consider underserved communities.",
                    "Assess the distribution of service and burdens.",
                    "Support customer accessibility.",
                ],
                "metrics": ["service_equity", "community_impact", "accessibility_score"],
            },
            EthicalPrinciple.ECONOMIC_VIABILITY: {
                "description": "Represent operational cost and resource efficiency.",
                "constraints": [
                    "Monitor operating costs.",
                    "Assess resource utilization.",
                    "Consider long-term financial sustainability.",
                ],
                "metrics": ["cost_efficiency", "resource_utilization"],
            },
            EthicalPrinciple.TRANSPARENCY: {
                "description": "Make indicators, assumptions, and reasoning visible.",
                "constraints": [
                    "Document the basis for routing decisions.",
                    "Make stakeholder-impact measures visible.",
                    "Maintain an audit record.",
                ],
                "metrics": ["explanation_quality", "audit_completeness"],
            },
            EthicalPrinciple.ACCOUNTABILITY: {
                "description": "Track compliance and provide feedback mechanisms.",
                "constraints": [
                    "Track compliance indicators.",
                    "Record stakeholder feedback.",
                    "Document recommendations.",
                ],
                "metrics": ["compliance_rate", "feedback_response"],
            },
        }
        self.ethical_constraints = []
        self.compliance_history = deque(maxlen=1000)
        self.audit_trail = []

    def add_ethical_constraint(
        self,
        principle: EthicalPrinciple,
        description: str,
        constraint_type: str,
        threshold: float,
        penalty_weight: float = 1.0,
    ) -> None:
        """Register a threshold rule used by the penalty calculation."""
        if constraint_type not in {"hard", "soft", "penalty"}:
            raise ValueError("constraint_type must be 'hard', 'soft', or 'penalty'.")
        if not 0.0 <= threshold <= 1.0:
            raise ValueError("threshold must be between 0 and 1.")
        if penalty_weight < 0:
            raise ValueError("penalty_weight must be non-negative.")

        self.ethical_constraints.append(
            EthicalConstraint(
                constraint_id=len(self.ethical_constraints),
                principle=principle,
                description=description,
                constraint_type=constraint_type,
                threshold=threshold,
                penalty_weight=penalty_weight,
            )
        )

    def evaluate_ethical_compliance(
        self, solution: ValueAlignedSolution
    ) -> Dict[EthicalPrinciple, float]:
        """Map solution indicators to bounded, illustrative compliance scores."""
        metrics = solution.accountability_metrics
        raw_scores = {
            EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: (
                max(0.0, 1.0 - solution.environmental_impact / 100.0)
                + metrics.get("renewable_energy", 0.5)
            )
            / 2.0,
            EthicalPrinciple.SOCIAL_EQUITY: (
                metrics.get("service_equity", 0.5) + solution.social_impact
            )
            / 2.0,
            EthicalPrinciple.ECONOMIC_VIABILITY: (
                max(0.0, 1.0 - solution.economic_cost / 1000.0)
                + metrics.get("cost_efficiency", 0.5)
            )
            / 2.0,
            EthicalPrinciple.TRANSPARENCY: (
                solution.transparency_score
                + metrics.get("audit_completeness", 0.5)
            )
            / 2.0,
            EthicalPrinciple.ACCOUNTABILITY: (
                metrics.get("compliance_rate", 0.5)
                + metrics.get("feedback_response", 0.5)
            )
            / 2.0,
        }

        # Principles without a dedicated indicator receive an explicit neutral placeholder.
        for principle in EthicalPrinciple:
            raw_scores.setdefault(principle, 0.7)
        return {
            principle: min(1.0, max(0.0, score))
            for principle, score in raw_scores.items()
        }

    def calculate_ethical_penalty(self, solution: ValueAlignedSolution) -> float:
        """Return the configured soft penalties or infinity for a hard violation."""
        compliance_scores = self.evaluate_ethical_compliance(solution)
        total_penalty = 0.0

        for constraint in self.ethical_constraints:
            compliance = compliance_scores[constraint.principle]
            if compliance >= constraint.threshold:
                continue
            if constraint.constraint_type == "hard":
                return float("inf")
            if constraint.constraint_type == "soft":
                total_penalty += (
                    constraint.threshold - compliance
                ) * constraint.penalty_weight
            elif constraint.constraint_type == "penalty":
                total_penalty += (1.0 - compliance) * constraint.penalty_weight

        return total_penalty

    def generate_ethical_reasoning(self, solution: ValueAlignedSolution) -> List[str]:
        """Describe the score bands used by this illustrative assessment."""
        scores = self.evaluate_ethical_compliance(solution)
        statements = []
        descriptions = {
            EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: "environmental sustainability",
            EthicalPrinciple.SOCIAL_EQUITY: "social equity",
            EthicalPrinciple.ECONOMIC_VIABILITY: "economic viability",
        }

        for principle, label in descriptions.items():
            score = scores[principle]
            if score > 0.8:
                assessment = "scores strongly under the stated indicators"
            elif score > 0.6:
                assessment = "shows a moderate score under the stated indicators"
            else:
                assessment = "has a low score under the stated indicators"
            statements.append(f"{label.capitalize()} {assessment} ({score:.2f}).")

        statements.append(
            f"The reported transparency indicator is {solution.transparency_score:.2f}."
        )
        statements.append(
            "Stakeholder weights and proxy measures should be reviewed with affected groups."
        )
        return statements


constitutional_ai = ConstitutionalAIFramework()
constitutional_ai.add_ethical_constraint(
    EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY,
    "Environmental compliance should meet the illustrative threshold.",
    "soft",
    0.7,
    penalty_weight=2.0,
)
constitutional_ai.add_ethical_constraint(
    EthicalPrinciple.SOCIAL_EQUITY,
    "Service equity should meet the illustrative threshold.",
    "soft",
    0.6,
    penalty_weight=1.5,
)
constitutional_ai.add_ethical_constraint(
    EthicalPrinciple.ECONOMIC_VIABILITY,
    "Economic viability should meet the illustrative threshold.",
    "soft",
    0.5,
    penalty_weight=1.0,
)

print(
    f"Created an illustrative framework with "
    f"{len(constitutional_ai.ethical_constraints)} ethical constraints."
)
print(
    f"The framework defines {len(constitutional_ai.constitution)} "
    "principle descriptions; other principles use a neutral placeholder score."
)

## Stakeholder utility model

A stakeholder utility is a score constructed from indicators that matter to that stakeholder. A weight \(w_s\) controls how much that stakeholder's utility contributes to the weighted total; the individual utility remains available for reporting. The utility functions below are linear, concave, convex, or threshold-based. These functions encode illustrative preferences, not measured human values.

For example, a linear utility is \(u(x)=ax+b\), a concave utility is \(u(x)=a\sqrt{x}\), a convex utility is \(u(x)=ax^2\), and a step utility is \(u(x)=v\) when \(x\) reaches its threshold and zero otherwise. All indicators supplied to these functions are intended to lie between zero and one.

In [ ]:
class StakeholderUtilityOptimizer:
    """Calculates weighted stakeholder utility and builds assessed solutions."""

    def __init__(
        self,
        stakeholders: List[Stakeholder],
        constitutional_ai: ConstitutionalAIFramework,
    ):
        self.stakeholders = stakeholders
        self.constitutional_ai = constitutional_ai
        self.stakeholder_weights = {
            stakeholder.stakeholder_id: stakeholder.weight
            for stakeholder in stakeholders
        }
        self.utility_functions = {
            "linear": lambda x, params: params.get("slope", 1.0) * x
            + params.get("intercept", 0.0),
            "concave": lambda x, params: params.get("scale", 1.0)
            * np.sqrt(max(0.0, x)),
            "convex": lambda x, params: params.get("scale", 1.0) * x**2,
            "step": lambda x, params: params.get("value", 1.0)
            if x >= params.get("threshold", 0.5)
            else 0.0,
        }
        self.optimization_history = deque(maxlen=100)
        self.utility_evolution = []

    def calculate_stakeholder_utility(
        self,
        solution_metrics: Dict[str, float],
        stakeholder: Stakeholder,
    ) -> float:
        """Calculate one stakeholder's utility from the supplied indicators."""
        utility_function = self.utility_functions.get(
            stakeholder.utility_function, self.utility_functions["linear"]
        )
        preferences = stakeholder.preferences

        if stakeholder.stakeholder_type == StakeholderType.CUSTOMER:
            indicators = [
                ("service_quality", "service_weight"),
                ("timeliness", "time_weight"),
                ("cost_affordability", "cost_weight"),
            ]
            return sum(
                utility_function(
                    solution_metrics.get(metric, 0.5),
                    {"slope": preferences.get(weight, 1.0)},
                )
                for metric, weight in indicators
            )

        if stakeholder.stakeholder_type == StakeholderType.ENVIRONMENT:
            values = [
                (1.0 - solution_metrics.get("emissions_level", 0.5), "emissions_weight"),
                (solution_metrics.get("renewable_energy", 0.5), "renewable_weight"),
                (solution_metrics.get("energy_efficiency", 0.5), "efficiency_weight"),
            ]
            return sum(
                utility_function(value, {"scale": preferences.get(weight, 1.0)})
                for value, weight in values
            )

        if stakeholder.stakeholder_type == StakeholderType.COMMUNITY:
            values = [
                ("local_employment", "jobs_weight"),
                ("traffic_reduction", "traffic_weight"),
                ("accessibility", "access_weight"),
            ]
            return sum(
                utility_function(
                    solution_metrics.get(metric, 0.5),
                    {"slope": preferences.get(weight, 1.0)},
                )
                for metric, weight in values
            )

        if stakeholder.stakeholder_type == StakeholderType.EMPLOYEE:
            values = [
                ("job_security", "security_weight"),
                ("wage_fairness", "wage_weight"),
                ("working_conditions", "conditions_weight"),
            ]
            return sum(
                utility_function(
                    solution_metrics.get(metric, 0.5),
                    {
                        "value": preferences.get(weight, 1.0),
                        "threshold": preferences.get("threshold", 0.5),
                    },
                )
                for metric, weight in values
            )

        if stakeholder.stakeholder_type == StakeholderType.INVESTOR:
            values = [
                ("profitability", "profit_weight"),
                ("growth_potential", "growth_weight"),
                ("risk_management", "risk_weight"),
            ]
            return sum(
                utility_function(value, {"scale": preferences.get(weight, 1.0)})
                for value, weight in (
                    (solution_metrics.get(metric, 0.5), weight)
                    for metric, weight in values
                )
            )

        if stakeholder.stakeholder_type == StakeholderType.REGULATOR:
            values = [
                ("regulatory_compliance", "compliance_weight"),
                ("safety_score", "safety_weight"),
                ("public_interest", "public_weight"),
            ]
            return sum(
                utility_function(
                    solution_metrics.get(metric, 0.5),
                    {"slope": preferences.get(weight, 1.0)},
                )
                for metric, weight in values
            )

        if stakeholder.stakeholder_type == StakeholderType.SOCIETY:
            values = [
                ("social_welfare", "welfare_weight"),
                ("sustainability", "sustainability_weight"),
                ("technological_progress", "progress_weight"),
            ]
            return sum(
                utility_function(
                    solution_metrics.get(metric, 0.5),
                    {"scale": preferences.get(weight, 1.0)},
                )
                for metric, weight in values
            )

        return 0.0

    def calculate_total_utility(self, solution_metrics: Dict[str, float]) -> float:
        """Sum stakeholder utilities after applying their decision weights."""
        return sum(
            self.calculate_stakeholder_utility(solution_metrics, stakeholder)
            * stakeholder.weight
            for stakeholder in self.stakeholders
        )

    def generate_value_aligned_solution(
        self,
        routes: List[List[Tuple[float, float]]],
        solution_metrics: Dict[str, float],
    ) -> ValueAlignedSolution:
        """Build a solution record and attach its compliance and reasoning scores."""
        stakeholder_utilities = {
            stakeholder.stakeholder_id: self.calculate_stakeholder_utility(
                solution_metrics, stakeholder
            )
            for stakeholder in self.stakeholders
        }
        solution = ValueAlignedSolution(
            routes=routes,
            stakeholder_utilities=stakeholder_utilities,
            ethical_compliance={},
            total_utility=self.calculate_total_utility(solution_metrics),
            environmental_impact=solution_metrics.get("emissions_level", 0.0)
            * 100.0,
            social_impact=solution_metrics.get("social_impact", 0.0),
            economic_cost=solution_metrics.get("total_cost", 0.0),
            transparency_score=solution_metrics.get("transparency", 0.0),
            accountability_metrics=solution_metrics,
            ethical_reasoning=[],
        )
        solution.ethical_compliance = (
            self.constitutional_ai.evaluate_ethical_compliance(solution)
        )
        solution.ethical_reasoning = (
            self.constitutional_ai.generate_ethical_reasoning(solution)
        )
        return solution


stakeholders = [
    Stakeholder(
        1, StakeholderType.CUSTOMER, "Customer Association", "linear", 0.25,
        {"service_weight": 0.4, "time_weight": 0.3, "cost_weight": 0.3},
    ),
    Stakeholder(
        2, StakeholderType.ENVIRONMENT, "Environmental Protection Agency",
        "concave", 0.20,
        {"emissions_weight": 0.5, "renewable_weight": 0.3, "efficiency_weight": 0.2},
    ),
    Stakeholder(
        3, StakeholderType.COMMUNITY, "Community Council", "linear", 0.15,
        {"jobs_weight": 0.3, "traffic_weight": 0.4, "access_weight": 0.3},
    ),
    Stakeholder(
        4, StakeholderType.EMPLOYEE, "Employee Union", "step", 0.15,
        {
            "security_weight": 0.4, "wage_weight": 0.3,
            "conditions_weight": 0.3, "threshold": 0.6,
        },
    ),
    Stakeholder(
        5, StakeholderType.INVESTOR, "Investment Group", "convex", 0.15,
        {"profit_weight": 0.5, "growth_weight": 0.3, "risk_weight": 0.2},
    ),
    Stakeholder(
        6, StakeholderType.REGULATOR, "Transportation Authority", "linear", 0.05,
        {"compliance_weight": 0.4, "safety_weight": 0.4, "public_weight": 0.2},
    ),
    Stakeholder(
        7, StakeholderType.SOCIETY, "Public Interest Group", "concave", 0.05,
        {"welfare_weight": 0.4, "sustainability_weight": 0.3, "progress_weight": 0.3},
    ),
]

assert np.isclose(sum(stakeholder.weight for stakeholder in stakeholders), 1.0)
utility_optimizer = StakeholderUtilityOptimizer(stakeholders, constitutional_ai)
print(f"Created a stakeholder utility model for {len(stakeholders)} stakeholders.")
for stakeholder in stakeholders:
    print(
        f"  {stakeholder.name}: weight {stakeholder.weight:.2f}; "
        f"type {stakeholder.stakeholder_type.value}"
    )

## Route construction and metric definitions

The heuristic uses a **greedy choice**: at each step it selects the highest-scoring feasible remaining customer. This is simple to trace, but a locally best choice can affect which customers remain feasible later. The battery check includes both the leg to the proposed customer and a direct return to the depot, so the route does not depend on unmodeled charging. Customer identities are retained while customers are removed from the unserved list; this keeps each customer's social-importance and sensitivity values attached to the correct location.

For a route of total distance \(D\), energy use is \(E=0.8D\) kWh, emissions are \(0.5E\) kg CO\(_2\), and illustrative cost is \(0.1D+0.15E\) currency units. These formulas are deliberately simple and are applied consistently to every route leg, including the return to the depot.

In [ ]:
class EthicalEVRouter:
    """Construct battery-feasible EV routes with a transparent ethical score."""

    def __init__(self, utility_optimizer: StakeholderUtilityOptimizer):
        self.utility_optimizer = utility_optimizer
        self.depot_location = (0.0, 0.0)
        self.customer_locations = [
            (15, 10), (20, 18), (8, 22), (25, 6), (12, 15), (18, 25)
        ]
        self.charging_stations = [(8, 12), (22, 20)]
        self.battery_capacity = 100.0
        self.energy_consumption_rate = 0.8
        self.reserve_energy = 20.0
        self.max_range = self.battery_capacity / self.energy_consumption_rate
        self.social_importance = [0.8, 0.6, 0.9, 0.5, 0.7, 0.4]
        self.renewable_availability = [0.7, 0.9]
        self.community_sensitivity = [0.3, 0.2, 0.8, 0.1, 0.5, 0.4]
        self.solution_history = []
        self.ethical_performance = []

    def calculate_distance(
        self,
        point1: Tuple[float, float],
        point2: Tuple[float, float],
    ) -> float:
        """Return Euclidean distance in the coordinate system's kilometre units."""
        return float(np.hypot(point1[0] - point2[0], point1[1] - point2[1]))

    def calculate_route_metrics(
        self, route: List[Tuple[float, float]]
    ) -> Dict[str, float]:
        """Calculate distance, energy, emissions, cost, and route proxies."""
        if len(route) < 2:
            return {
                "total_distance": 0.0,
                "energy_consumed": 0.0,
                "emissions": 0.0,
                "cost": 0.0,
                "social_impact": 0.0,
                "renewable_usage": 0.0,
            }

        distances = [
            self.calculate_distance(route[index], route[index + 1])
            for index in range(len(route) - 1)
        ]
        total_distance = sum(distances)
        energy_consumed = total_distance * self.energy_consumption_rate
        emissions = energy_consumed * 0.5
        cost = total_distance * 0.1 + energy_consumed * 0.15

        visited_customers = [
            self.customer_locations.index(point)
            for point in route[1:-1]
            if point in self.customer_locations
        ]
        social_impact = (
            float(np.mean([self.social_importance[index] for index in visited_customers]))
            if visited_customers
            else 0.0
        )

        nearby_station_scores = []
        for station_index, station in enumerate(self.charging_stations):
            if any(self.calculate_distance(point, station) < 5.0 for point in route):
                nearby_station_scores.append(self.renewable_availability[station_index])
        renewable_opportunity = (
            float(np.mean(nearby_station_scores)) if nearby_station_scores else 0.0
        )

        return {
            "total_distance": total_distance,
            "energy_consumed": energy_consumed,
            "emissions": emissions,
            "cost": cost,
            "social_impact": social_impact,
            "renewable_usage": renewable_opportunity,
        }

    def generate_ethical_routes(self) -> List[List[Tuple[float, float]]]:
        """Greedily assign feasible customers to at most two depot-return routes."""
        remaining_customers = list(enumerate(self.customer_locations))
        routes = []

        for _vehicle in range(2):
            if not remaining_customers:
                break

            route = [self.depot_location]
            current_location = self.depot_location
            battery_level = self.battery_capacity

            while remaining_customers:
                best_position = None
                best_score = -float("inf")

                for position, (customer_index, customer) in enumerate(remaining_customers):
                    distance = self.calculate_distance(current_location, customer)
                    return_distance = self.calculate_distance(customer, self.depot_location)
                    energy_needed = (distance + return_distance) * self.energy_consumption_rate
                    if energy_needed > battery_level - self.reserve_energy:
                        continue

                    social_score = self.social_importance[customer_index] * 0.4
                    efficiency_score = (1.0 / (1.0 + distance)) * 0.3
                    community_score = (
                        1.0 - self.community_sensitivity[customer_index]
                    ) * 0.3
                    total_score = social_score + efficiency_score + community_score

                    if total_score > best_score:
                        best_score = total_score
                        best_position = position

                if best_position is None:
                    break

                customer_index, customer = remaining_customers.pop(best_position)
                route.append(customer)
                battery_level -= (
                    self.calculate_distance(current_location, customer)
                    * self.energy_consumption_rate
                )
                current_location = customer

            if len(route) > 1:
                return_energy = (
                    self.calculate_distance(current_location, self.depot_location)
                    * self.energy_consumption_rate
                )
                if battery_level - return_energy < self.reserve_energy - 1e-9:
                    raise AssertionError("A constructed route violates the battery reserve.")
                route.append(self.depot_location)
                routes.append(route)

        if remaining_customers:
            unserved_ids = [customer_index + 1 for customer_index, _ in remaining_customers]
            raise RuntimeError(
                "The two-vehicle heuristic could not serve every customer while "
                f"preserving the battery reserve. Unserved customer IDs: {unserved_ids}."
            )

        served_customers = [point for route in routes for point in route[1:-1]]
        assert len(served_customers) == len(self.customer_locations)
        assert len(set(served_customers)) == len(self.customer_locations)
        return routes

    def optimize_ethical_routing(self) -> ValueAlignedSolution:
        """Construct routes, evaluate their indicators, and subtract soft penalties."""
        print("Constructing routes with the score-based heuristic...")
        routes = self.generate_ethical_routes()

        total_metrics = {
            "total_distance": 0.0,
            "energy_consumed": 0.0,
            "emissions_level": 0.0,
            "total_cost": 0.0,
            "social_impact": 0.0,
            "renewable_energy": 0.0,
            "service_quality": 0.8,
            "timeliness": 0.85,
            "cost_affordability": 0.4,
            "local_employment": 0.7,
            "traffic_reduction": 0.6,
            "accessibility": 0.8,
            "job_security": 0.9,
            "wage_fairness": 0.8,
            "working_conditions": 0.85,
            "profitability": 0.7,
            "growth_potential": 0.6,
            "risk_management": 0.8,
            "regulatory_compliance": 0.95,
            "safety_score": 0.9,
            "public_interest": 0.8,
            "social_welfare": 0.75,
            "sustainability": 0.8,
            "technological_progress": 0.7,
            "transparency": 0.85,
            "energy_efficiency": 0.8,
            "cost_efficiency": 0.75,
            "resource_utilization": 0.8,
            "explanation_quality": 0.9,
            "audit_completeness": 0.85,
            "reasoning_transparency": 0.8,
            "compliance_rate": 0.9,
            "feedback_response": 0.8,
            "violation_resolution": 0.85,
            "service_equity": 0.8,
            "community_impact": 0.7,
        }

        route_metrics = [self.calculate_route_metrics(route) for route in routes]
        for metrics in route_metrics:
            total_metrics["total_distance"] += metrics["total_distance"]
            total_metrics["energy_consumed"] += metrics["energy_consumed"]
            total_metrics["emissions_level"] += metrics["emissions"]
            total_metrics["total_cost"] += metrics["cost"]
            total_metrics["renewable_energy"] += metrics["renewable_usage"]

        total_metrics["social_impact"] = float(
            np.mean(self.social_importance)
        )
        total_metrics["emissions_level"] = min(
            1.0, total_metrics["emissions_level"] / 100.0
        )
        total_metrics["renewable_energy"] /= max(1, len(routes))

        solution = self.utility_optimizer.generate_value_aligned_solution(
            routes, total_metrics
        )
        ethical_penalty = self.utility_optimizer.constitutional_ai.calculate_ethical_penalty(
            solution
        )
        if np.isinf(ethical_penalty):
            solution.total_utility = 0.0
        else:
            solution.total_utility -= ethical_penalty

        self.solution_history.append(solution)
        self.ethical_performance.append(solution.ethical_compliance)
        return solution


ethical_router = EthicalEVRouter(utility_optimizer)
print(
    f"Router data: {len(ethical_router.customer_locations)} customers, "
    f"{len(ethical_router.charging_stations)} mapped charging stations, "
    "and two vehicles."
)

## Construct and inspect the route plan

The route count and metrics below are computed when the notebook is run. A route is a list of coordinates beginning and ending at the depot. `total_utility` is the weighted stakeholder utility **after** the configured ethical penalty; the displayed penalty is reported separately so its effect remains interpretable. The battery check applies to the complete depot-return route, while charging-station proximity remains only a proxy.

In [ ]:
print("Ethical EV Route Construction")
print("=" * 40)
ethical_solution = ethical_router.optimize_ethical_routing()

print("\nRoute-plan summary:")
print(f"Number of routes: {len(ethical_solution.routes)}")
print(f"Adjusted weighted utility: {ethical_solution.total_utility:.3f}")
print(
    "Environmental impact: "
    f"{ethical_solution.environmental_impact:.2f} kg CO2 (illustrative model)"
)
print(f"Social-importance score: {ethical_solution.social_impact:.3f}")
print(f"Illustrative operating cost: {ethical_solution.economic_cost:.2f}")
print(f"Transparency indicator: {ethical_solution.transparency_score:.2f}")

print("\nStakeholder utilities:")
stakeholders_by_id = {
    stakeholder.stakeholder_id: stakeholder for stakeholder in stakeholders
}
for stakeholder_id, utility in ethical_solution.stakeholder_utilities.items():
    print(f"  {stakeholders_by_id[stakeholder_id].name}: {utility:.3f}")

print("\nIllustrative compliance indicators:")
for principle, compliance in ethical_solution.ethical_compliance.items():
    if compliance > 0.8:
        status = "high under this scoring rule"
    elif compliance > 0.6:
        status = "moderate under this scoring rule"
    else:
        status = "low under this scoring rule"
    print(f"  {principle.value}: {compliance:.3f} ({status})")

print("\nAssessment notes:")
for index, reasoning in enumerate(ethical_solution.ethical_reasoning, start=1):
    print(f"  {index}. {reasoning}")

ethical_penalty = constitutional_ai.calculate_ethical_penalty(ethical_solution)
if np.isinf(ethical_penalty):
    print("\nEthical penalty: infeasible under a configured hard constraint.")
else:
    print(f"\nConfigured ethical penalty: {ethical_penalty:.3f}")

print("\nRoute details:")
for route_number, route in enumerate(ethical_solution.routes, start=1):
    metrics = ethical_router.calculate_route_metrics(route)
    distance = metrics["total_distance"]
    energy = metrics["energy_consumed"]
    assert energy <= ethical_router.battery_capacity - ethical_router.reserve_energy + 1e-9
    print(f"\nRoute {route_number}:")
    print(f"  Number of listed locations, including depot endpoints: {len(route)}")
    print(f"  Distance: {distance:.2f} km")
    print(f"  Energy: {energy:.2f} kWh")
    print(f"  Emissions: {metrics['emissions']:.2f} kg CO2")
    print(f"  Illustrative operating cost: {metrics['cost']:.2f}")
    print(
        "  Renewable-charging opportunity proxy: "
        f"{metrics['renewable_usage']:.1%}"
    )
    print(f"  Mean social-importance score of served customers: {metrics['social_impact']:.2f}")

## Comparison with an illustrative utilitarian reference

The reference below is a hand-specified scenario whose indicators are passed through the same stakeholder and compliance scoring functions. It is not generated by a profit-maximizing routing algorithm, and its route-related values are not a second feasible route plan. The comparison therefore illustrates how the scoring framework reports different indicator profiles; it cannot establish that one routing method outperforms another.

In [ ]:
def compare_with_utilitarian_approach(
    ethical_solution: ValueAlignedSolution,
    stakeholders: List[Stakeholder],
    utility_optimizer: StakeholderUtilityOptimizer,
    constitutional_ai: ConstitutionalAIFramework,
) -> Dict:
    """Compare the solution with a synthetic, profit-focused reference profile."""
    reference_metrics = {
        "total_distance": 120.0,
        "energy_consumed": 96.0,
        "emissions_level": 0.48,
        "total_cost": 300.0,
        "social_impact": 0.30,
        "renewable_energy": 0.20,
        "service_quality": 0.6,
        "timeliness": 0.7,
        "cost_affordability": 0.7,
        "local_employment": 0.4,
        "traffic_reduction": 0.3,
        "accessibility": 0.6,
        "job_security": 0.7,
        "wage_fairness": 0.6,
        "working_conditions": 0.7,
        "profitability": 0.9,
        "growth_potential": 0.8,
        "risk_management": 0.6,
        "regulatory_compliance": 0.7,
        "safety_score": 0.7,
        "public_interest": 0.5,
        "social_welfare": 0.4,
        "sustainability": 0.3,
        "technological_progress": 0.6,
        "transparency": 0.4,
        "energy_efficiency": 0.6,
        "cost_efficiency": 0.9,
        "resource_utilization": 0.9,
        "explanation_quality": 0.3,
        "audit_completeness": 0.4,
        "reasoning_transparency": 0.3,
        "compliance_rate": 0.7,
        "feedback_response": 0.4,
        "violation_resolution": 0.5,
        "service_equity": 0.4,
        "community_impact": 0.3,
    }

    reference_utilities = {
        stakeholder.stakeholder_id: utility_optimizer.calculate_stakeholder_utility(
            reference_metrics, stakeholder
        )
        for stakeholder in stakeholders
    }
    reference_weighted_utility = sum(
        reference_utilities[stakeholder.stakeholder_id] * stakeholder.weight
        for stakeholder in stakeholders
    )
    reference_solution = ValueAlignedSolution(
        routes=[],
        stakeholder_utilities=reference_utilities,
        ethical_compliance={},
        total_utility=reference_weighted_utility,
        environmental_impact=reference_metrics["emissions_level"] * 100.0,
        social_impact=reference_metrics["social_impact"],
        economic_cost=reference_metrics["total_cost"],
        transparency_score=reference_metrics["transparency"],
        accountability_metrics=reference_metrics,
        ethical_reasoning=[],
    )
    reference_solution.ethical_compliance = (
        constitutional_ai.evaluate_ethical_compliance(reference_solution)
    )

    reference_weighted_utility -= constitutional_ai.calculate_ethical_penalty(
        reference_solution
    )

    print("Illustrative reference-profile comparison")
    print("=" * 48)
    print("The reference is a synthetic indicator profile, not an optimized route plan.")
    print(f"{'Metric':<30}{'Constructed plan':>18}{'Reference':>14}")
    print("-" * 62)
    comparison_rows = [
        ("Adjusted weighted utility", ethical_solution.total_utility, reference_weighted_utility),
        ("Emissions (kg CO2)", ethical_solution.environmental_impact, reference_solution.environmental_impact),
        ("Social-importance score", ethical_solution.social_impact, reference_solution.social_impact),
        ("Illustrative operating cost", ethical_solution.economic_cost, reference_solution.economic_cost),
        (
            "Renewable opportunity proxy",
            ethical_solution.accountability_metrics.get("renewable_energy", 0.0),
            reference_metrics["renewable_energy"],
        ),
        ("Transparency indicator", ethical_solution.transparency_score, reference_solution.transparency_score),
    ]
    for label, plan_value, reference_value in comparison_rows:
        print(f"{label:<30}{plan_value:>18.3f}{reference_value:>14.3f}")

    print("\nStakeholder utility scores:")
    for stakeholder in stakeholders:
        stakeholder_id = stakeholder.stakeholder_id
        print(
            f"  {stakeholder.name}: "
            f"plan={ethical_solution.stakeholder_utilities[stakeholder_id]:.3f}, "
            f"reference={reference_utilities[stakeholder_id]:.3f}"
        )

    print("\nCompliance-score comparison:")
    for principle in EthicalPrinciple:
        print(
            f"  {principle.value}: "
            f"plan={ethical_solution.ethical_compliance[principle]:.3f}, "
            f"reference={reference_solution.ethical_compliance[principle]:.3f}"
        )

    return {
        "ethical_solution": ethical_solution,
        "utilitarian_solution": reference_metrics,
        "ethical_utilities": ethical_solution.stakeholder_utilities,
        "utilitarian_utilities": reference_utilities,
        "utilitarian_compliance": reference_solution.ethical_compliance,
        "utilitarian_weighted_utility": reference_weighted_utility,
    }


comparison_results = compare_with_utilitarian_approach(
    ethical_solution,
    stakeholders,
    utility_optimizer,
    constitutional_ai,
)

## Visual interpretation

The route map distinguishes the depot, customers, charging stations, and vehicle routes. Customer marker size represents the supplied social-importance value, while the station labels show renewable availability. The station labels describe modeled availability, not charging performed by the route. The charts compare the constructed plan with the synthetic reference profile using the same stated indicator conventions.

In [ ]:
def visualize_ethical_ev_routing(
    ethical_solution: ValueAlignedSolution,
    comparison_results: Dict,
    ethical_router: EthicalEVRouter,
    stakeholders: List[Stakeholder],
) -> None:
    """Plot routes, stakeholder scores, compliance scores, and a metric summary."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    route_ax, utility_ax, compliance_ax, summary_ax = axes.flat
    fig.suptitle("EV Routing: Value and Compliance Indicators", fontsize=16)

    route_colors = ["tab:blue", "tab:green"]
    for route_index, route in enumerate(ethical_solution.routes):
        route_ax.plot(
            [point[0] for point in route],
            [point[1] for point in route],
            marker="o",
            linewidth=2,
            color=route_colors[route_index % len(route_colors)],
            label=f"Vehicle route {route_index + 1}",
        )

    for index, station in enumerate(ethical_router.charging_stations):
        renewable_pct = ethical_router.renewable_availability[index] * 100.0
        route_ax.scatter(
            station[0], station[1], marker="s", s=180, color="gold",
            edgecolor="black", label="Charging station" if index == 0 else None,
        )
        route_ax.annotate(
            f"{renewable_pct:.0f}% available", station,
            xytext=(5, 5), textcoords="offset points", fontsize=8,
        )

    for index, customer in enumerate(ethical_router.customer_locations):
        importance = ethical_router.social_importance[index]
        route_ax.scatter(
            customer[0], customer[1], marker="^",
            s=80 + importance * 160, color="tab:red", alpha=0.75,
            edgecolor="black", label="Customer" if index == 0 else None,
        )
        route_ax.annotate(
            f"C{index + 1} ({importance:.1f})", customer,
            xytext=(4, 4), textcoords="offset points", fontsize=8,
        )

    depot = ethical_router.depot_location
    route_ax.scatter(
        depot[0], depot[1], marker="D", s=220, color="black",
        edgecolor="gold", label="Depot",
    )
    route_ax.set_title("Constructed routes and locations")
    route_ax.set_xlabel("X coordinate (km)")
    route_ax.set_ylabel("Y coordinate (km)")
    route_ax.set_aspect("equal", adjustable="datalim")
    route_ax.grid(True, alpha=0.3)
    route_ax.legend(fontsize=8)

    names = [stakeholder.name for stakeholder in stakeholders]
    plan_values = [
        ethical_solution.stakeholder_utilities[stakeholder.stakeholder_id]
        for stakeholder in stakeholders
    ]
    reference_values = [
        comparison_results["utilitarian_utilities"][stakeholder.stakeholder_id]
        for stakeholder in stakeholders
    ]
    positions = np.arange(len(names))
    width = 0.38
    utility_ax.bar(
        positions - width / 2, plan_values, width,
        label="Constructed plan", color="lightblue",
    )
    utility_ax.bar(
        positions + width / 2, reference_values, width,
        label="Synthetic reference", color="lightcoral",
    )
    utility_ax.set_title("Stakeholder utility scores")
    utility_ax.set_ylabel("Utility score (model units)")
    utility_ax.set_xlabel("Stakeholder")
    utility_ax.set_xticks(positions)
    utility_ax.set_xticklabels(names, rotation=40, ha="right", fontsize=8)
    utility_ax.legend(fontsize=8)
    utility_ax.grid(axis="y", alpha=0.3)

    principles = list(ethical_solution.ethical_compliance)
    labels = [principle.value.replace("_", "\n") for principle in principles]
    plan_scores = [ethical_solution.ethical_compliance[p] for p in principles]
    reference_scores = [
        comparison_results["utilitarian_compliance"][p] for p in principles
    ]
    positions = np.arange(len(principles))
    compliance_ax.bar(
        positions - width / 2, plan_scores, width,
        label="Constructed plan", color="tab:blue",
    )
    compliance_ax.bar(
        positions + width / 2, reference_scores, width,
        label="Synthetic reference", color="tab:red",
    )
    compliance_ax.set_title("Illustrative compliance indicators")
    compliance_ax.set_ylabel("Score (0 to 1)")
    compliance_ax.set_xlabel("Principle")
    compliance_ax.set_ylim(0, 1)
    compliance_ax.set_xticks(positions)
    compliance_ax.set_xticklabels(labels, fontsize=7)
    compliance_ax.legend(fontsize=8)
    compliance_ax.grid(axis="y", alpha=0.3)

    summary = (
        "Constructed route plan\n"
        f"Adjusted weighted utility: {ethical_solution.total_utility:.3f}\n"
        f"Emissions: {ethical_solution.environmental_impact:.2f} kg CO2\n"
        f"Social-importance score: {ethical_solution.social_impact:.3f}\n"
        f"Illustrative cost: {ethical_solution.economic_cost:.2f}\n"
        f"Transparency indicator: {ethical_solution.transparency_score:.2f}\n\n"
        "Interpretation limits\n"
        "The reference is a synthetic profile, not a route solution.\n"
        "Station proximity is an opportunity proxy, not charging data.\n"
        "Scores depend on the stated illustrative assumptions."
    )
    summary_ax.text(
        0.04, 0.96, summary, transform=summary_ax.transAxes,
        va="top", ha="left", fontsize=10, family="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightyellow", "alpha": 0.8},
    )
    summary_ax.set_title("Metric summary and scope")
    summary_ax.axis("off")

    fig.tight_layout()
    plt.show()


visualize_ethical_ev_routing(
    ethical_solution, comparison_results, ethical_router, stakeholders
)

## Ethical audit

An audit record aggregates the displayed compliance indicators, lists configured threshold violations, summarizes stakeholder utility bands, and derives recommendations from the indicators. The improvement field is computed only when a reference compliance profile is supplied; it is a relative score difference, not evidence of real-world ethical improvement. Feedback categories in this example are generated from utility scores and are not responses collected from actual stakeholders.

In [ ]:
def create_ethical_audit_report(
    solution: ValueAlignedSolution,
    stakeholders: List[Stakeholder],
    constitutional_ai: ConstitutionalAIFramework,
    baseline_compliance: Optional[Dict[EthicalPrinciple, float]] = None,
) -> EthicalAudit:
    """Create an audit summary from supplied solution and framework data."""
    compliance_values = list(solution.ethical_compliance.values())
    overall_compliance = float(np.mean(compliance_values)) if compliance_values else 0.0
    constraints_by_principle = {
        constraint.principle: constraint
        for constraint in constitutional_ai.ethical_constraints
    }
    violations = []
    for principle, constraint in constraints_by_principle.items():
        score = solution.ethical_compliance[principle]
        if score < constraint.threshold:
            violations.append(
                f"{principle.value}: score {score:.3f} is below "
                f"the configured threshold {constraint.threshold:.3f}."
            )

    feedback = {}
    for stakeholder in stakeholders:
        utility = solution.stakeholder_utilities[stakeholder.stakeholder_id]
        if utility > 0.8:
            category = "High utility under the illustrative scoring rule."
        elif utility > 0.6:
            category = "Moderate-to-high utility under the illustrative scoring rule."
        elif utility > 0.4:
            category = "Moderate utility under the illustrative scoring rule."
        else:
            category = "Low utility under the illustrative scoring rule."
        feedback[stakeholder.stakeholder_id] = category

    recommendations = []
    if solution.ethical_compliance.get(
        EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY, 0.0
    ) < 0.7:
        recommendations.append(
            "Review the renewable-charging opportunity proxy and emissions assumptions."
        )
        recommendations.append(
            "Assess route changes that reduce distance without violating battery limits."
        )
    if solution.ethical_compliance.get(EthicalPrinciple.SOCIAL_EQUITY, 0.0) < 0.7:
        recommendations.append(
            "Review service-equity indicators with affected communities."
        )
        recommendations.append(
            "Examine whether customer importance values represent underserved groups."
        )
    if solution.ethical_compliance.get(EthicalPrinciple.TRANSPARENCY, 0.0) < 0.8:
        recommendations.append(
            "Document the assumptions, indicator sources, and route-scoring rationale."
        )
    if solution.ethical_compliance.get(EthicalPrinciple.ACCOUNTABILITY, 0.0) < 0.8:
        recommendations.append(
            "Define a process for collecting and responding to stakeholder feedback."
        )
    if not recommendations:
        recommendations.append(
            "Continue monitoring the stated indicators and review their assumptions."
        )

    relative_improvement = 0.0
    if baseline_compliance:
        shared_principles = set(solution.ethical_compliance) & set(baseline_compliance)
        if shared_principles:
            baseline_mean = float(
                np.mean([baseline_compliance[p] for p in shared_principles])
            )
            solution_mean = float(
                np.mean([solution.ethical_compliance[p] for p in shared_principles])
            )
            if not np.isclose(baseline_mean, 0.0):
                relative_improvement = (
                    (solution_mean - baseline_mean) / abs(baseline_mean) * 100.0
                )

    timestamp = datetime.now()
    audit = EthicalAudit(
        audit_id=f"audit_{timestamp.strftime('%Y%m%d_%H%M%S')}",
        timestamp=timestamp,
        solution=solution,
        compliance_score=overall_compliance,
        violations=violations,
        stakeholder_feedback=feedback,
        recommendations=recommendations,
        ethical_score_improvement=relative_improvement,
    )
    constitutional_ai.audit_trail.append(audit)
    constitutional_ai.compliance_history.append(overall_compliance)

    print("Ethical audit summary")
    print("=" * 40)
    print(f"Audit ID: {audit.audit_id}")
    print(f"Timestamp: {audit.timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Mean illustrative compliance score: {audit.compliance_score:.3f}")
    if baseline_compliance:
        print(
            "Relative compliance-score difference from the supplied reference: "
            f"{audit.ethical_score_improvement:+.1f}%"
        )
        print("This is a score comparison, not a measured ethical improvement.")

    if audit.violations:
        print(f"\nConfigured threshold violations ({len(audit.violations)}):")
        for violation in audit.violations:
            print(f"  - {violation}")
    else:
        print("\nConfigured threshold violations: none.")

    print("\nUtility-derived stakeholder categories (not collected feedback):")
    stakeholder_names = {
        stakeholder.stakeholder_id: stakeholder.name for stakeholder in stakeholders
    }
    for stakeholder_id, category in audit.stakeholder_feedback.items():
        print(f"  {stakeholder_names[stakeholder_id]}: {category}")

    print("\nRecommendations:")
    for index, recommendation in enumerate(audit.recommendations, start=1):
        print(f"  {index}. {recommendation}")
    return audit


ethical_audit = create_ethical_audit_report(
    ethical_solution,
    stakeholders,
    constitutional_ai,
    comparison_results["utilitarian_compliance"],
)

## Method strengths, limitations, and appropriate use

### What this example demonstrates

- Stakeholder preferences can be represented by explicit utility functions and weights.
- Soft ethical thresholds can be translated into transparent penalties.
- A route-construction heuristic can include social and community indicators in its selection score while enforcing a battery-reserve check.
- Reporting separate metrics, assumptions, and audit notes makes the assessment easier to inspect.

### Limitations

- The route constructor is greedy. It does not enumerate all route assignments, prove optimality, or guarantee that it finds a feasible plan whenever one exists.
- The ethical score is used to rank feasible next-customer choices; stakeholder utility and soft penalties are calculated after route construction. Consequently, the route is not the maximizer of the reported adjusted weighted utility.
- Stakeholder weights, utility functions, normalization constants, and the synthetic reference profile are illustrative choices. Real applications require stakeholder engagement, evidence-based indicators, and governance over how values are represented.
- The renewable opportunity proxy does not model charging events, energy sources, or station capacity. Compliance indicators are not legal determinations or proof of ethical conduct.
- The simplified emissions and cost formulas omit important real-world effects, including electricity-generation mix, vehicle manufacturing, road conditions, time, and changing energy prices.

### When the modeling ideas may be useful

The framework of explicit stakeholders, interpretable indicators, and documented constraints can help structure discussion in public services, regulated operations, and logistics systems with substantial social or environmental effects. The numerical model here is a teaching example, not a deployment-ready decision system.

### Pros and trade-offs

| Aspect | Potential benefit | Important limitation |
|---|---|---|
| Stakeholder representation | Makes selected preferences and weights visible | Chosen weights may omit or misrepresent affected groups |
| Soft ethical thresholds | Makes trade-offs and penalties explicit | Scores and thresholds require justification and review |
| Greedy route construction | Each local choice is easy to inspect | Local choices do not guarantee a globally best plan |
| Audit reporting | Organizes indicators, thresholds, and recommendations | A report cannot establish ethical legitimacy by itself |
| Illustrative comparison | Shows how profiles can be assessed consistently | A synthetic reference is not an empirical or algorithmic benchmark |

### Further study within this modeling framework

Natural extensions include adding time windows, customer demand, station capacity, charging decisions, and measured emissions or cost data while retaining a clearly specified route-construction and assessment process. Any such extension would require revisiting feasibility checks, units, indicator definitions, and the interpretation of the resulting scores.